In [0]:
%pip install kafka-python

In [0]:
import json 
import random
import time
from datetime import datetime
from kafka import KafkaProducer
import ssl

#Get secrets 

In [0]:
host = dbutils.secrets.get("jdbc", "kafka-host")
port = dbutils.secrets.get("jdbc", "kafka-port")
topic = dbutils.secrets.get("jdbc", "kafka-topic")

#Write certificates to temp files 

In [0]:
import tempfile, os 

tmpdir = tempfile.mkdtemp()

with open(f"{tmpdir}/service.key", "w") as f:
  f.write(dbutils.secrets.get("jdbc", "kafka-access-key"))

with open(f"{tmpdir}/service.cert", "w") as f:
  f.write(dbutils.secrets.get("jdbc", "kafka-access-cert"))

with open(f"{tmpdir}/ca.pem", "w") as f:
  f.write(dbutils.secrets.get("jdbc", "kafka-ca-cert"))
  


#SSL context

In [0]:
ssl_ctx = ssl.create_default_context()
ssl_ctx.load_verify_locations(f"{tmpdir}/ca.pem")
ssl_ctx.load_cert_chain(
    certfile=f"{tmpdir}/service.cert",
    keyfile=f"{tmpdir}/service.key"
)

#Create Producer

In [0]:
producer = KafkaProducer(
    bootstrap_servers=f"{host}:{port}",
    security_protocol="SSL",
    ssl_context=ssl_ctx,
    value_serializer=lambda v: json.dumps(v).encode('utf-8')
)

#Generate and send 50 fake sales events

In [0]:
products  = ["BK-M18B-40", "BK-R50B-44", "BK-T44U-46", "BK-M82B-38"]
customers = [11000, 11001, 11002, 11003, 11004]

for i in range(50):
    event = {
        "order_id": f"50-STREAM-{i+1:04d}",
        "customer_id": random.choice(customers),
        "product_id": random.choice(products),
        "quantity": random.randint(1, 5),
        "price": round(random.uniform(500, 3000), 2),
        "event_time": datetime.utcnow().isoformat()
    }
    producer.send(topic, value=event)
    print(f"Sent: {event}")
    time.sleep(0.1)

producer.flush()
producer.close()
print("Done - 50 events sent to Kafka")